In [2]:
from datetime import datetime, timedelta
import os
import random
import numpy as np
import pandas as pd

# Set reproducible seeds
random.seed(42)
np.random.seed(42)

# Ensure raw data folder exists relative to this notebook
os.makedirs("../data/raw", exist_ok=True)

# ----------------------------------------------------
# 1. GENERATE: dim_lead_sources
# ----------------------------------------------------
lead_sources_data = [
    {
        "lead_source_id": "LS-01",
        "channel_name": "Developer RFP",
        "cost_per_lead_aed": 1200,
    },
    {
        "lead_source_id": "LS-02",
        "channel_name": "Architect Referral",
        "cost_per_lead_aed": 500,
    },
    {
        "lead_source_id": "LS-03",
        "channel_name": "Direct Inbound / Web",
        "cost_per_lead_aed": 350,
    },
    {
        "lead_source_id": "LS-04",
        "channel_name": "Repeat Customer",
        "cost_per_lead_aed": 0,
    },
]
df_lead_sources = pd.DataFrame(lead_sources_data)

# ----------------------------------------------------
# 2. GENERATE: dim_clients
# ----------------------------------------------------
locations = [
    "Palm Jumeirah",
    "Dubai Hills Estate",
    "Emirates Hills",
    "Arabian Ranches",
    "Downtown Dubai",
    "Business Bay",
    "Jumeirah Golf Estates",
]

client_types = [
    "Luxury Residential",
    "Commercial Developer",
    "Hospitality",
    "Corporate Office",
]

clients_list = []
NUM_CLIENTS = 120

for i in range(1, NUM_CLIENTS + 1):
    c_type = random.choice(client_types)
    if c_type == "Luxury Residential":
        name = f"Private Client {i:03d}"
    elif c_type == "Hospitality":
        name = f"Resort & Spa Group {i:03d}"
    else:
        name = f"Al-{random.choice(['Futtaim', 'Ghurair', 'Habtoor', 'Diyar'])} Holdings {i:03d}"

    clients_list.append(
        {
            "client_id": f"CL-{1000 + i}",
            "client_name": name,
            "client_type": c_type,
            "primary_location": random.choice(locations),
        }
    )
df_clients = pd.DataFrame(clients_list)

# ----------------------------------------------------
# 3. GENERATE: fact_projects
# ----------------------------------------------------
client_ids = df_clients["client_id"].tolist()
lead_ids = df_lead_sources["lead_source_id"].tolist()

project_types = {
    "Full Villa Landscape": (2500, 8000, (65, 110)),
    "Commercial Plaza": (6000, 20000, (45, 80)),
    "Pool & Hardscape": (1000, 3500, (120, 190)),
    "Irrigation & Softscape": (1500, 5000, (30, 55)),
}

projects_list = []
NUM_PROJECTS = 300
START_DATE_RANGE = datetime(2023, 1, 1)

for i in range(1, NUM_PROJECTS + 1):
    p_id = f"PRJ-{2023 + (i // 150)}-{i:03d}"
    p_type = random.choice(list(project_types.keys()))
    client = random.choice(client_ids)
    lead = random.choice(lead_ids)

    min_area, max_area, rate_range = project_types[p_type]
    area_sqft = random.randint(min_area, max_area)
    price_per_sqft = random.uniform(rate_range[0], rate_range[1])

    contract_value = round((area_sqft * price_per_sqft) / 500) * 500
    planned_margin = random.uniform(0.28, 0.35)
    estimated_budget = round(contract_value * (1 - planned_margin), -2)

    duration_days = int(area_sqft / random.randint(45, 75)) + random.randint(
        15, 30
    )
    start_offset = random.randint(0, 600)
    start_date = START_DATE_RANGE + timedelta(days=start_offset)
    planned_end = start_date + timedelta(days=duration_days)

    is_summer = start_date.month in [6, 7, 8, 9]
    delay_chance = 0.55 if is_summer else 0.25
    is_delayed = random.random() < delay_chance

    extra_days = random.randint(10, 45) if is_delayed else random.randint(-5, 0)
    actual_end = planned_end + timedelta(days=extra_days)
    status = "Completed" if actual_end < datetime(2025, 1, 1) else "In Progress"

    projects_list.append(
        {
            "project_id": p_id,
            "client_id": client,
            "lead_source_id": lead,
            "project_type": p_type,
            "area_sqft": area_sqft,
            "contract_value_aed": contract_value,
            "estimated_budget_aed": estimated_budget,
            "start_date": start_date.strftime("%Y-%m-%d"),
            "planned_end_date": planned_end.strftime("%Y-%m-%d"),
            "actual_end_date": (
                actual_end.strftime("%Y-%m-%d")
                if status == "Completed"
                else None
            ),
            "status": status,
        }
    )
df_projects = pd.DataFrame(projects_list)

# ----------------------------------------------------
# 4. EXPORT TO CSV
# ----------------------------------------------------
df_lead_sources.to_csv("../data/raw/dim_lead_sources.csv", index=False)
df_clients.to_csv("../data/raw/dim_clients.csv", index=False)
df_projects.to_csv("../data/raw/fact_projects.csv", index=False)

print("Status: 3 tables generated and saved to ../data/raw/")

Status: 3 tables generated and saved to ../data/raw/


In [3]:
import os
import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

# Load the fact_projects table generated in the previous step
df_projects = pd.read_csv("../data/raw/fact_projects.csv")

# ----------------------------------------------------
# 1. GENERATE: fact_project_costs
# ----------------------------------------------------
costs_list = []

for idx, row in df_projects.iterrows():
    p_id = row["project_id"]
    budget = row["estimated_budget_aed"]
    status = row["status"]

    # If project is still in progress, costs logged to date are partial
    cost_multiplier = 1.0
    if status == "In Progress":
        cost_multiplier = random.uniform(0.35, 0.75)
    else:
        # Completed projects: simulate budget performance
        scenario = random.choices(
            ["on_budget", "minor_overrun", "severe_overrun"],
            weights=[0.70, 0.20, 0.10],
            k=1,
        )[0]

        if scenario == "on_budget":
            cost_multiplier = random.uniform(0.92, 1.03)
        elif scenario == "minor_overrun":
            cost_multiplier = random.uniform(1.05, 1.18)
        else:
            cost_multiplier = random.uniform(1.22, 1.45)

    actual_total = round(budget * cost_multiplier, -2)

    # Cost breakdown allocations
    mat_cost = round(actual_total * random.uniform(0.38, 0.44), -2)
    lab_cost = round(actual_total * random.uniform(0.22, 0.28), -2)
    sub_cost = round(actual_total * random.uniform(0.16, 0.24), -2)
    eqp_cost = round(actual_total * random.uniform(0.07, 0.12), -2)
    permit_cost = (
        actual_total - (mat_cost + lab_cost + sub_cost + eqp_cost)
    )  # Balances to exact total

    costs_list.append(
        {
            "cost_entry_id": f"CST-{idx + 1:04d}",
            "project_id": p_id,
            "material_cost_aed": mat_cost,
            "labour_cost_aed": lab_cost,
            "subcontractor_cost_aed": sub_cost,
            "equipment_rental_aed": eqp_cost,
            "permitting_fees_aed": permit_cost,
            "actual_total_cost_aed": actual_total,
        }
    )

df_costs = pd.DataFrame(costs_list)

# ----------------------------------------------------
# 2. GENERATE: fact_maintenance (AMCs)
# ----------------------------------------------------
# Merge client info to determine AMC propensity
df_clients = pd.read_csv("../data/raw/dim_clients.csv")
merged_proj = df_projects.merge(df_clients, on="client_id", how="left")

maintenance_list = []
m_counter = 1

for idx, row in merged_proj.iterrows():
    if row["status"] != "Completed":
        continue

    # Higher conversion for Residential and Hospitality
    conversion_prob = (
        0.65
        if row["client_type"] in ["Luxury Residential", "Hospitality"]
        else 0.25
    )

    if random.random() < conversion_prob:
        start_dt = pd.to_datetime(row["actual_end_date"]) + pd.Timedelta(
            days=14
        )
        end_dt = start_dt + pd.Timedelta(days=365)

        # Annual maintenance fee (~8% to 14% of original installation contract value)
        annual_fee = round(
            (row["contract_value_aed"] * random.uniform(0.08, 0.14)) / 250
        ) * 250
        monthly_cost = round((annual_fee * random.uniform(0.50, 0.65)) / 12, -1)

        renewal = random.choices(
            ["Renewed", "Active", "Expired"], weights=[0.45, 0.40, 0.15], k=1
        )[0]

        maintenance_list.append(
            {
                "contract_id": f"AMC-{m_counter:04d}",
                "project_id": row["project_id"],
                "client_id": row["client_id"],
                "start_date": start_dt.strftime("%Y-%m-%d"),
                "end_date": end_dt.strftime("%Y-%m-%d"),
                "annual_fee_aed": annual_fee,
                "monthly_operating_cost_aed": monthly_cost,
                "renewal_status": renewal,
            }
        )
        m_counter += 1

df_maintenance = pd.DataFrame(maintenance_list)

# ----------------------------------------------------
# 3. EXPORT ADDITIONAL CSVs
# ----------------------------------------------------
df_costs.to_csv("../data/raw/fact_project_costs.csv", index=False)
df_maintenance.to_csv("../data/raw/fact_maintenance.csv", index=False)

print("Status: Successfully generated and saved to data/raw/:")
print(f"- fact_project_costs.csv ({len(df_costs)} records)")
print(f"- fact_maintenance.csv ({len(df_maintenance)} records)")

Status: Successfully generated and saved to data/raw/:
- fact_project_costs.csv (300 records)
- fact_maintenance.csv (125 records)
